# 음성 챗봇

## 화면 구성

In [ ]:
import gradio as gr
import requests
import datetime
import dotenv
import os
dotenv.load_dotenv()
OPEN_AI_KEY2 = os.getenv('OPEN_AI_KEY2')
AZURE_SPEECH_KEY = os.getenv('AZURE_SPEECH_KEY')

############ STT ##############
def request_stt(audio_path):
    endpoint = "https://eastus.stt.speech.microsoft.com/speech/recognition/conversation/cognitiveservices/v1?language=ko-KR&format=detailed"
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
        "Content-Type": "audio/wav"
    }
    with open(audio_path, 'rb') as audio_file:
        audio_data = audio_file.read()

    # Path로부터 오디오 데이터를 읽어와서 data에 추가하는로직을 작성해야 합니다.

    response = requests.post(endpoint, headers=headers, data=audio_data)
    print(response)

    if not response.ok:
        return None
    
    response_json = response.json()
    display_text = response_json['NBest'][0]['Display']

    return display_text

############ TTS #####################
def request_tts(input_text):
    endpoint = "https://eastus.tts.speech.microsoft.com/cognitiveservices/v1"

    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": "riff-8khz-16bit-mono-pcm"
        }
    
    body = f"""
        <speak version='1.0' xml:lang='en-US'>
            <voice xml:lang='ko-KR' xml:gender='Female' name='	ko-KR-SunHi:DragonHDLatestNeural'>
                <!-- my voice is <break strength="medium" /> my passport verify me -->
                {input_text}

            </voice>
        </speak>
    """
    
    response = requests.post(endpoint, headers=headers, data=body)

    if not response.ok:
        print(f"Error: {response.status_code} - {response.text}")
        return None
    
    now = datetime.datetime.now()
    file_name = "tts_{}.wav".format(now.strftime("%Y%m%d_%H%M%S"))

    with open(file_name, "wb") as audio_file:
        audio_file.write(response.content)

    print(response)

    return file_name

############# Open AI ###################
def request_openai(prompt, histories=[]):
    import requests
    import os 
    import dotenv
    dotenv.load_dotenv()
    OPEN_AI_KEY2 = os.getenv("OPEN_AI_KEY2")

    # Endpoint: OpenAI 호출하기 위한 Endpoint URI, URL
    endpoint = "https://fimtrus-foundry.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
 

    # Method : Post

    # Header: OpenAI API Key 포함된 헤더 정보
    headers = {
        # "api-key": OPEN_AI_KEY,
        "Authorization": OPEN_AI_KEY2,
        "Content-Type": "application/json"
    }

    ###### 이전 메시지 반영 코드 추가 ######

    messsage_list = list()

    # 시스템 메시지를 메시지 리스트에 추가
    messsage_list.append({
        "role":"system", 
        "content":  "## 역할\n당신은 '보물섬 남해'를 전문적으로 안내하는 친절하고 유능한 관광 가이드입니다. 제공된 남해 관광지 데이터를 바탕으로 사용자의 여행 계획을 돕고 정보를 제공합니다.\n\n## 답변 원칙\n1. **데이터 우선주의**: 반드시 제공된 검색 결과(Context)에 있는 정보만을 바탕으로 답변하세요. 데이터에 없는 내용은 추측하여 지어내지 마세요.\n2. **구체적 정보 활용**: 답변 시 관광지명, 주소, 테마, 주차 가능 여부(has_parkinglot) 정보를 적극적으로 언급하여 실질적인 도움을 주세요.\n3. **태도**: 남해의 따뜻하고 아름다운 이미지를 전달할 수 있도록 친절하고 환대하는 말투를 사용하세요.\n4. **모르는 경우**: 데이터에 해당 관광지 정보가 없다면 \"죄송하지만, 현재 제가 가진 남해 관광 데이터에는 해당 장소에 대한 정보가 없습니다.\"라고 정직하게 답변하세요.\n\n## 답변 형식 (예시)\n- 추천 시: \"[관광지명]을 추천해 드려요! 이곳은 [테마] 테마의 장소로, [주소]에 위치해 있습니다. (주차 가능 여부 언급)\"\n- 상세 설명: 데이터의 'description' 내용을 요약하여 흥미롭게 전달하세요."
    })

    # 히스토리가 존재하는 경우 메시지 리스트에 모든 데이터를 추가
    for history in histories:
        # content가 텍스트가 아닌 경우
        role = history['role']
        content = history['content'][0]['text']
        messsage_list.append({"role": role, "content": content})

        # content가 텍스트인 경우 - 그대로 넣기
        # messsage_list.append(history)
        
    # 사용자 프롬프트 메시지를 메시지 리스트에 추가한다
    messsage_list.append({
        "role":"user", 
        "content": prompt
    })

    # Body: Body의 메시지 정보
    body = { 
            "messages": messsage_list,
            "max_tokens": 4096,
            "temperature": 0.7,
            "top_p": 0.95,
            "model": "9ai043-gpt-4o-mini"
            }

    # OpenAI API 호출
    response = requests.post(endpoint, headers=headers, json=body)
    response_json = response.json()

    content = response_json['choices'][0]['message']['content']
    role = response_json['choices'][0]['message']['role']
    print(role, content)
    return {"role": role, "content": content}


with gr.Blocks() as demo:
    ############# Event Listener ############
    def stop_recording(audio_path):
        print("STOP_RECORDING")
        print(audio_path)
        display_text = request_stt(audio_path)
        print(display_text)
        
        return display_text

    def click_button(input_text):
        print('버튼이 클릭되었습니다!')
        print(f'입력된 텍스트: {input_text}')
        file_name = request_tts(input_text)
        return file_name
    
    def click_send(prompt, histories):
        assistant_data = request_openai(prompt)
        # histories = list()
        histories.append({"role": "user", "content": prompt})
        histories.append(assistant_data)
        print(histories)
        return histories


    with gr.Row():
        ########## Open AI ################
        with gr.Column(scale=5):
            chatbot = gr.Chatbot(label='남해 관광지 챗봇')
            with gr.Row():
                prompt_audio = gr.Audio(label="질문", sources="microphone", type="filepath", scale=1)
                input_textbox = gr.Textbox(label="질문을 입력하세요", scale = 5)
                send_button = gr.Button("전송", scale=1)
            openai_audio = gr.Audio(
                label='질문', interactive=False, autoplay=True
                
            )


        ###################################
        
        with gr.Column(scale=1):
            ############# STT #################
            with gr.Column():
                gr.Markdown("### Speech To Text")
                input_audio = gr.Audio(label="음성을 입력해주세요", sources = "microphone", type="filepath")
                output_text = gr.Textbox(label="음성 인식 결과", placeholder="여기에 음성 인식 결과가 표시됩니다.", interactive=False)
            ###################################

            ############# TTS #################
            with gr.Column():
                gr.Markdown("### Text To Speech")
                input_text = gr.Textbox(label="변환할 텍스트 입력")
                send_button = gr.Button("전송")
                output_audio = gr.Audio(label="음성 출력", interactive=False, autoplay=True)

            ###################################
    input_audio.stop_recording(stop_recording, inputs=[input_audio], outputs=[output_text])
    send_button.click(click_button, inputs=[input_text], outputs=[output_audio])
    send_button.click(fn=click_send, inputs=[input_textbox, chatbot], outputs= [chatbot])

demo.launch()
# stop_recording(r"C:\Users\USER\Downloads\도깨비.wav")
# history_list = [{'role': 'user', 'metadata': None, 'content': [{'text': '안녕', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': '안녕하세요! 어떻게 도와드릴까요?', 'type': 'text'}], 'options': None}, {'role': 'user', 'content': '반가워'}, {'role': 'assistant', 'content': '반가워요! 어떻게 도와드릴까요?'}]
# click_send("남해에서 가볼만한 곳이 어디야?", history_list)

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import gradio as gr
import requests
import datetime
import dotenv
import os
dotenv.load_dotenv()
OPEN_AI_KEY2 = os.getenv('OPEN_AI_KEY2')
AZURE_SPEECH_KEY = os.getenv('AZURE_SPEECH_KEY')

############ STT ##############
def request_stt(audio_path):
    endpoint = "https://eastus.stt.speech.microsoft.com/speech/recognition/conversation/cognitiveservices/v1?language=ko-KR&format=detailed"
    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
        "Content-Type": "audio/wav"
    }
    with open(audio_path, 'rb') as audio_file:
        audio_data = audio_file.read()

    # Path로부터 오디오 데이터를 읽어와서 data에 추가하는로직을 작성해야 합니다.

    response = requests.post(endpoint, headers=headers, data=audio_data)
    print(response)

    if not response.ok:
        return None
    
    response_json = response.json()
    display_text = response_json['NBest'][0]['Display']

    return display_text

############ TTS #####################
def request_tts(input_text):
    endpoint = "https://eastus.tts.speech.microsoft.com/cognitiveservices/v1"

    headers = {
        "Ocp-Apim-Subscription-Key": AZURE_SPEECH_KEY,
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": "riff-8khz-16bit-mono-pcm"
        }
    
    body = f"""
        <speak version='1.0' xml:lang='en-US'>
            <voice xml:lang='ko-KR' xml:gender='Female' name='	ko-KR-SunHi:DragonHDLatestNeural'>
                <!-- my voice is <break strength="medium" /> my passport verify me -->
                {input_text}

            </voice>
        </speak>
    """
    
    response = requests.post(endpoint, headers=headers, data=body)

    if not response.ok:
        print(f"Error: {response.status_code} - {response.text}")
        return None
    
    now = datetime.datetime.now()
    file_name = "tts_{}.wav".format(now.strftime("%Y%m%d_%H%M%S"))

    with open(file_name, "wb") as audio_file:
        audio_file.write(response.content)

    print(response)

    return file_name

############# Open AI ###################
def request_openai(prompt, histories=[]):
    import requests
    import os 
    import dotenv
    dotenv.load_dotenv()
    OPEN_AI_KEY2 = os.getenv("OPEN_AI_KEY2")

    # Endpoint: OpenAI 호출하기 위한 Endpoint URI, URL
    endpoint = "https://fimtrus-foundry.cognitiveservices.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview"
 

    # Method : Post

    # Header: OpenAI API Key 포함된 헤더 정보
    headers = {
        # "api-key": OPEN_AI_KEY,
        "Authorization": OPEN_AI_KEY2,
        "Content-Type": "application/json"
    }

    ###### 이전 메시지 반영 코드 추가 ######

    messsage_list = list()

    # 시스템 메시지를 메시지 리스트에 추가
    messsage_list.append({
        "role":"system", 
        "content":  "## 역할\n당신은 '보물섬 남해'를 전문적으로 안내하는 친절하고 유능한 관광 가이드입니다. 제공된 남해 관광지 데이터를 바탕으로 사용자의 여행 계획을 돕고 정보를 제공합니다.\n\n## 답변 원칙\n1. **데이터 우선주의**: 반드시 제공된 검색 결과(Context)에 있는 정보만을 바탕으로 답변하세요. 데이터에 없는 내용은 추측하여 지어내지 마세요.\n2. **구체적 정보 활용**: 답변 시 관광지명, 주소, 테마, 주차 가능 여부(has_parkinglot) 정보를 적극적으로 언급하여 실질적인 도움을 주세요.\n3. **태도**: 남해의 따뜻하고 아름다운 이미지를 전달할 수 있도록 친절하고 환대하는 말투를 사용하세요.\n4. **모르는 경우**: 데이터에 해당 관광지 정보가 없다면 \"죄송하지만, 현재 제가 가진 남해 관광 데이터에는 해당 장소에 대한 정보가 없습니다.\"라고 정직하게 답변하세요.\n\n## 답변 형식 (예시)\n- 추천 시: \"[관광지명]을 추천해 드려요! 이곳은 [테마] 테마의 장소로, [주소]에 위치해 있습니다. (주차 가능 여부 언급)\"\n- 상세 설명: 데이터의 'description' 내용을 요약하여 흥미롭게 전달하세요."
    })

    # 히스토리가 존재하는 경우 메시지 리스트에 모든 데이터를 추가
    for history in histories:
        # content가 텍스트가 아닌 경우
        role = history['role']
        content = history['content'][0]['text']
        messsage_list.append({"role": role, "content": content})

        # content가 텍스트인 경우 - 그대로 넣기
        # messsage_list.append(history)
        
    # 사용자 프롬프트 메시지를 메시지 리스트에 추가한다
    messsage_list.append({
        "role":"user", 
        "content": prompt
    })

    # Body: Body의 메시지 정보
    body = { 
            "messages": messsage_list,
            "max_tokens": 4096,
            "temperature": 0.7,
            "top_p": 0.95,
            "model": "9ai043-gpt-4o-mini"
            }

    # OpenAI API 호출
    response = requests.post(endpoint, headers=headers, json=body)
    response_json = response.json()

    content = response_json['choices'][0]['message']['content']
    role = response_json['choices'][0]['message']['role']
    print(role, content)
    return {"role": role, "content": content}


with gr.Blocks() as demo:
    ############# Event Listener ############
    def stop_recording(audio_path):
        print("STOP_RECORDING")
        print(audio_path)
        display_text = request_stt(audio_path)
        print(display_text)
        
        return display_text

    def click_button(input_text):
        print('버튼이 클릭되었습니다!')
        print(f'입력된 텍스트: {input_text}')
        file_name = request_tts(input_text)
        return file_name
    
    def click_send(prompt, histories):
        assistant_data = request_openai(prompt, histories)
        # histories = list()
        if assistant_data is None:
            return histories
        
        histories.append({"role": "user", "content": prompt})
        histories.append(assistant_data)
        print(histories)
        return histories

    def click_send_voice(prompt, histories):
        assistant_data = request_openai(prompt, histories)
        print(histories)
        if assistant_data is None:
            return histories
        histories.append({"role": "user", "content": prompt})
        histories.append(assistant_data)

        return histories


    with gr.Row():
        ########## Open AI ################
        with gr.Column(scale=5):
            chatbot = gr.Chatbot(label='남해 관광지 챗봇')
            with gr.Row():
                prompt_audio = gr.Audio(label="질문", sources="microphone", type="filepath", scale=1)
                input_textbox = gr.Textbox(label="질문을 입력하세요", scale = 5)
                # chatbot_send_button = gr.Button("전송", scale=1)
            openai_audio = gr.Audio(
                label='질문', interactive=False, autoplay=True
                
            )


        ###################################
        
        with gr.Column(scale=1):
            ############# STT #################
            with gr.Column():
                gr.Markdown("### Speech To Text")
                input_audio = gr.Audio(label="음성을 입력해주세요", sources = "microphone", type="filepath")
                output_text = gr.Textbox(label="음성 인식 결과", placeholder="여기에 음성 인식 결과가 표시됩니다.", interactive=False)
            ###################################

            ############# TTS #################
            with gr.Column():
                gr.Markdown("### Text To Speech")
                input_text = gr.Textbox(label="변환할 텍스트 입력")
                send_button = gr.Button("전송")
                output_audio = gr.Audio(label="음성 출력", interactive=False, autoplay=True)

            ###################################
    input_audio.stop_recording(stop_recording, inputs=[input_audio], outputs=[output_text])
    send_button.click(click_button, inputs=[input_text], outputs=[output_audio])
    # chatbot_send_button.click(fn=click_send, inputs=[input_textbox, chatbot], outputs= [chatbot])

    ## 질문 음성으로 받는 함수
    def stop_recording_openai(audio_path):
        display_text = request_stt(audio_path)
        return display_text
    

    ## 음성으로 받은 질문을 던지는 함수
    def change_input_textbox(input_text, histories):
        assistant_data = request_openai(input_text, histories)
        print(histories)
        if assistant_data is None:
            return histories
        histories.append({"role": "user", "content": input_text})
        histories.append(assistant_data)
        return histories

    ## 생성한 답변을 정제하는 함수
    def change_content(histories):
        if histories is None or len(histories)==0:
            return None
        content = histories[-1]["content"][0]["text"]
        pattern = r"[^가-힣a-zA-Z0-9\s!.,]"
        import re
        cleaned_content = re.sub(pattern,"", content)
        file_name = request_tts(cleaned_content)
        return file_name
    
    ## 질문 오디오 중지시 녹음 멈추기
    prompt_audio.stop_recording(
        stop_recording_openai, inputs=[prompt_audio], outputs=[input_textbox]
    )
    input_textbox.change(
        change_input_textbox, inputs=[input_textbox, chatbot], outputs=[chatbot]
    )

    chatbot.change(change_content, inputs=[chatbot], outputs=[openai_audio])

    
demo.launch()
# stop_recording(r"C:\Users\USER\Downloads\도깨비.wav")
# history_list = [{'role': 'user', 'metadata': None, 'content': [{'text': '안녕', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': '안녕하세요! 어떻게 도와드릴까요?', 'type': 'text'}], 'options': None}, {'role': 'user', 'content': '반가워'}, {'role': 'assistant', 'content': '반가워요! 어떻게 도와드릴까요?'}]
# click_send("남해에서 가볼만한 곳이 어디야?", history_list)

* Running on local URL:  http://127.0.0.1:7881
* To create a public link, set `share=True` in `launch()`.


버튼이 클릭되었습니다!
입력된 텍스트: 안녕하세요
<Response [200]>


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 410, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\sit

<Response [200]>
assistant 안녕하세요! 남해의 아름다움을 안내해드릴 관광 가이드입니다. 남해 여행이나 관광지에 대해 궁금한 점이 있으시면 언제든 말씀해 주세요. 따뜻하고 특별한 남해의 매력을 가득 담아 친절하게 안내해드리겠습니다!
[]
버튼이 클릭되었습니다!
입력된 텍스트: 남해관광지 추천해줘
<Response [200]>
<Response [200]>


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\uvicorn\protocols\http\h11_impl.py", line 410, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\starlette\applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\sit

<Response [200]>
assistant 네, 남해에서 멋진 여행을 즐기실 수 있도록 추천드릴 관광지를 안내해드릴게요!

**남해 독일마을**을 추천해 드려요! 이곳은 ‘마을’ 테마의 장소로, 경상남도 남해군 남해읍 남해대로 64에 위치해 있습니다. 주차장도 마련되어 있으니 편리하게 방문하실 수 있습니다.

남해 독일마을은 1960~70년대 독일에서 일하던 교포들이 귀국하여 만든 곳으로, 독일의 전통 건축 양식이 그대로 반영된 아름다운 마을이에요. 알록달록한 집들과 독일식 맥주를 맛볼 수 있는 곳이 있어, 이국적인 분위기를 만끽하실 수 있습니다. 남해 바다와 어우러진 독일마을의 풍경은 사진 찍기에도 정말 좋아요!

남해 여행에서 독일마을을 꼭 한 번 들러보시길 추천드립니다!
[{'role': 'user', 'metadata': None, 'content': [{'text': '안녕하세요.', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': '안녕하세요! 남해의 아름다움을 안내해드릴 관광 가이드입니다. 남해 여행이나 관광지에 대해 궁금한 점이 있으시면 언제든 말씀해 주세요. 따뜻하고 특별한 남해의 매력을 가득 담아 친절하게 안내해드리겠습니다!', 'type': 'text'}], 'options': None}]


Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connection_lost(None)>
Traceback (most recent call last):
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
  File "c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
ConnectionResetError: [WinError 10054] 현재 연결은 원격 호스트에 의해 강제로 끊겼습니다


<Response [200]>
assistant 안녕하세요! 남해 여행에 대해 궁금한 점이 있으신가요? 남해의 다양한 관광지를 안내해 드릴 수 있습니다. 원하시는 장소나 테마가 있으시면 말씀해 주세요!
[]
assistant 안녕하세요! 남해 여행을 준비 중이시라면 정말 좋은 선택이세요. 남해에는 아름다운 자연과 멋진 관광지가 많이 있답니다. 궁금한 관광지나 추천받고 싶은 곳이 있으시면 언제든 말씀해 주세요. 남해의 매력을 듬뿍 담아 안내해 드리겠습니다!
[{'role': 'user', 'metadata': None, 'content': [{'text': 'ㅇ', 'type': 'text'}], 'options': None}, {'role': 'assistant', 'metadata': None, 'content': [{'text': '안녕하세요! 남해 여행에 대해 궁금한 점이 있으신가요? 남해의 다양한 관광지를 안내해 드릴 수 있습니다. 원하시는 장소나 테마가 있으시면 말씀해 주세요!', 'type': 'text'}], 'options': None}]
<Response [200]>
<Response [200]>
버튼이 클릭되었습니다!
입력된 텍스트: 유산균은 언제 먹어야 좋아?
<Response [200]>
